# 배터리 + **점유 관측(Occupancy)** 환경 — Colab Runner (final 기반)

**무엇이 다른가**: `colab_runner_final.ipynb`와 동일한 흐름이지만, 관측(obs)에
**각 타깃칸의 점유 여부(occupancy) 벡터**를 추가한 정책을 학습/평가합니다.
드론이 "어느 칸이 비었는지"를 직접 보게 되어, 목표 한 칸 앞에서 얼어붙던
교착(funnel deadlock)을 완화하는 것이 목적입니다.

- 환경: `BatteryOccupancyShapeFormationEnv` (comm_env_occupancy.py) — 기존
  `BatteryShapeFormationEnv`를 상속해 obs 끝에 점유 벡터 n_agents개를 덧붙임.
  obs_dim이 n_agents만큼 커짐 (예: 14대 → 85 → 99).
- 학습: `comm_train_battery_occ.py` (기존 comm_train_battery.py를 env만 바꿔치기한 래퍼)
- 평가: `comm_eval_battery_occ.py`, sweep: `wind_sweep_battery_occ.py`
- **기존 비-점유 ckpt와 호환 안 됨** → 반드시 처음부터 학습. 산출물은 `_occ` 폴더로 분리 저장.
- 보상 개선도 함께 적용: assigned_target_reward↑, ent_coef↑, 상시 커버리지 보상(coverage_step_reward).

**런타임**: `런타임 → 런타임 유형 변경 → GPU (T4)` 필수.

## 1. GitHub clone

복구 시에도 이 셀만 다시 돌리면 OK. Drive 저장된 ckpt/log는 그대로 살아있음.

In [ ]:
BRANCH = "Saehoon"

%cd /content
!rm -rf /content/RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git /content/RL-2026s1-tp
%cd /content/RL-2026s1-tp

!git fetch origin
!git switch $BRANCH
!git pull origin $BRANCH

!git log --oneline -1
# 점유 관측 파일들이 받아졌는지 확인
!ls -l comm_env_occupancy.py comm_train_battery_occ.py comm_eval_battery_occ.py wind_sweep_battery_occ.py 2>/dev/null || echo '⚠️ 점유 관측 파일 없음 — 최신 코드 pull 확인 필요'
!grep -n 'coverage-step-reward' comm_train_battery.py | head -2 || echo '⚠️ coverage_step_reward 인자 없음 — 코드 동기화 확인'

## 2. Google Drive 마운트 + 저장 경로

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/drone_results/battery_letters"
!mkdir -p {DRIVE_ROOT}/ckpts {DRIVE_ROOT}/runs {DRIVE_ROOT}/gifs {DRIVE_ROOT}/evals {DRIVE_ROOT}/sweeps
print("DRIVE_ROOT =", DRIVE_ROOT)
!ls -la {DRIVE_ROOT}

## 3. 패키지 설치

In [ ]:
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

## 4. 학습 시퀀스 + 환경/학습/바람/보상 설정

`EXP_TAG`만 바꾸면 시퀀스별로 폴더가 분리됨. **점유 관측 산출물은 `_occ` 접미사로 비-점유와 분리 저장**.

**적용된 보상 개선** (얼어붙음 대응): `ASSIGNED_TARGET_REWARD` 0.1→0.4, `ENT_COEF` 0.01→0.02,
신규 `COVERAGE_STEP_REWARD`(상시 커버리지 보상).

In [ ]:
# ============================================================
# 학습/평가 공통 설정 (점유 관측 + 보상 개선)
# ============================================================
TARGET_SEQUENCE = "GROUND,D,G"        # ← 학습/평가 시퀀스 (자유 변경)
EXP_TAG         = "DG"                # ← Drive 폴더명에 들어감

# 환경
GRID_SIZE = 25
N_AGENTS  = 14
MAX_STEPS = 500

# 학습 hyperparam
TOTAL_FRAMES     = 800_000            # (축소) 1.2M→800k 상한. success 100% 도달 시 조기 종료되므로 보통 더 일찍 끝남
FRAMES_PER_BATCH = 4096
MINIBATCH_SIZE   = 512
PPO_EPOCHS       = 6
LR               = 2e-4
ENT_COEF         = 0.02               # (개선) 0.01 -> 0.02: 'stay' 국소최적 탈출
CLIP_EPS         = 0.15
CKPT_EVERY       = 5
EARLY_STOP_SUCCESS  = 1.0             # 배치 성공률이 이 값(1.0=100%)에 도달하면 조기 종료
EARLY_STOP_PATIENCE = 3               # 연속 N iter 충족 시 ckpt 저장 후 종료
SAVE_BEST_ABOVE     = 0.9             # 성공률 90% 초과 시, 직전 best보다 높으면 그 ckpt 저장(5의 배수 아니어도)

# Battery hyperparam
INITIAL_BATTERY          = 1.0
HOVER_BATTERY_COST       = 0.002
MOVE_BATTERY_COST        = 0.005
LOW_BATTERY_MOVE_PENALTY = 0.15

# Reward shaping
COMPLETION_REWARD       = 50.0
ASSIGNED_TARGET_REWARD  = 0.4         # (개선) 0.1 -> 0.4: 자기 칸 점유 가치↑ (캠핑 억제)
COVERAGE_DELTA_REWARD   = 0.3
COVERAGE_STEP_REWARD    = 0.01        # (신규) 매 스텝 점유칸 수 비례 보상: 빈칸=상시 손해
HOVER_PENALTY           = 0.05
SHAPING_COEF            = 0.5

# 바람 hyperparam
WIND_PROB      = 0.3
WIND_STRENGTH  = 1
EVAL_WIND_PROB = 0.3
WIND_LEVELS    = "0.0,0.1,0.2,0.3,0.4"

# A->B 전이학습 옵션
B_INIT_FROM_A           = True
B_FINETUNE_LR           = 1e-4
B_FINETUNE_ENT_COEF     = 0.01
B_FINETUNE_TOTAL_FRAMES = 500_000

# Drive 경로 — 점유 관측은 _occ 접미사로 분리 (비-점유 ckpt와 obs_dim이 달라 섞이면 안 됨)
# 평가/비교용 A/B는 통합 학습 매트릭스의 batt_base / batt_wind 케이스 ckpt를 가리킴
# (단독 A/B 학습 셀은 매트릭스와 중복이라 제거됨)
SAVE_DIR_A   = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_occ_batt_base"
SAVE_DIR_B   = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_occ_batt_wind"
TB_LOGDIR_A  = f"{DRIVE_ROOT}/runs/{EXP_TAG}_occ_batt_base"
TB_LOGDIR_B  = f"{DRIVE_ROOT}/runs/{EXP_TAG}_occ_batt_wind"
GIF_A        = f"{DRIVE_ROOT}/gifs/demo_{EXP_TAG}_battery_occ.gif"
GIF_B        = f"{DRIVE_ROOT}/gifs/demo_{EXP_TAG}_battery_wind_occ.gif"
EVAL_TXT_A   = f"{DRIVE_ROOT}/evals/eval_{EXP_TAG}_battery_occ.txt"
EVAL_TXT_B   = f"{DRIVE_ROOT}/evals/eval_{EXP_TAG}_battery_wind_occ.txt"
SWEEP_PNG    = f"{DRIVE_ROOT}/sweeps/wind_sweep_{EXP_TAG}_battery_occ.png"

print(f"TARGET_SEQUENCE = {TARGET_SEQUENCE}")
print(f"SAVE_DIR_A      = {SAVE_DIR_A}")
print(f"SAVE_DIR_B      = {SAVE_DIR_B}")
print(f"GIF_A / GIF_B   = {GIF_A} | {GIF_B}")

## 5. Smoke test — 점유 관측이 obs에 들어갔는지 확인

`obs_dim`이 비-점유 대비 **n_agents 만큼 커졌는지** 확인 (예: 14대 → 85 → 99).

In [ ]:
%cd /content/RL-2026s1-tp

from comm_env import BatteryShapeFormationEnv
from comm_env_occupancy import BatteryOccupancyShapeFormationEnv

shapes = [s.strip() for s in TARGET_SEQUENCE.split(',') if s.strip()]
kw = dict(grid_size=GRID_SIZE, n_agents=N_AGENTS, max_steps=MAX_STEPS, shapes=shapes,
          initial_battery=INITIAL_BATTERY, hover_battery_cost=HOVER_BATTERY_COST,
          move_battery_cost=MOVE_BATTERY_COST, low_battery_move_penalty=LOW_BATTERY_MOVE_PENALTY)

base_env = BatteryShapeFormationEnv(**kw)
occ_env  = BatteryOccupancyShapeFormationEnv(**kw)
obs, _ = occ_env.reset(seed=0)
one = obs[occ_env.possible_agents[0]]
print(f"base obs_dim         = {base_env.obs_dim}")
print(f"occupancy obs_dim    = {occ_env.obs_dim}  (= base + n_agents={N_AGENTS})")
print(f"actual obs length    = {len(one)}")
print(f"tail (occupancy) =", one[-N_AGENTS:])
assert occ_env.obs_dim == base_env.obs_dim + N_AGENTS and len(one) == occ_env.obs_dim
print("OK: 점유 벡터가 obs 끝에 정상 추가됨")

## 6. Resume helper — Drive에 있는 마지막 ckpt 찾기

In [ ]:
import os, re, glob

def latest_ckpt(save_dir):
    cand = []
    for p in glob.glob(os.path.join(save_dir, 'ckpt_*.pt')):
        m = re.search(r'ckpt_(\d+)\.pt$', p)
        if m:
            cand.append((int(m.group(1)), p))
    for d in glob.glob(f"{save_dir}_resume_from_*"):
        m = re.search(r'_resume_from_(\d+)$', d)
        if not m: continue
        base = int(m.group(1))
        for p in glob.glob(os.path.join(d, 'ckpt_*.pt')):
            mm = re.search(r'ckpt_(\d+)\.pt$', p)
            if mm:
                cand.append((base + int(mm.group(1)), p))
    if not cand:
        return None, 0
    cand.sort()
    return cand[-1][1], cand[-1][0]

A_RESUME_CKPT, A_RESUME_ITER = latest_ckpt(SAVE_DIR_A)
B_RESUME_CKPT, B_RESUME_ITER = latest_ckpt(SAVE_DIR_B)
for label, ck, it in [("A (occ)", A_RESUME_CKPT, A_RESUME_ITER),
                       ("B (occ+wind)", B_RESUME_CKPT, B_RESUME_ITER)]:
    print(f"[{label}] 이어 학습할 ckpt: {ck} (effective iter {it})" if ck else f"[{label}] ckpt 없음 -> 처음부터 학습")

## 7. (선택) 빠른 파이프라인 검증

In [ ]:
%cd /content/RL-2026s1-tp
!python comm_train_battery_occ.py --shapes GROUND,X --max-steps 250 \
  --total-frames 50000 --frames-per-batch 4096 \
  --wind-prob 0.3 --randomize-wind \
  --ckpt-every 5 --save-dir /tmp/ckpt_smoke_occ --tb-logdir /tmp/runs_smoke_occ

## 8. 통합 학습 — 배터리 유무 × 통신에러/바람 (셀 분리, 전부 warm-start)

배터리 {있음/없음} × 교란 {없음 / 바람 / 통신에러 / 바람+통신} 조합을 **케이스별 셀**로 나눠
하나씩 실행합니다(원하는 케이스만 골라 실행 가능).

- 먼저 아래 **공통 설정/헬퍼 셀**을 1번 실행 → `train_case(name)` 정의.
- 각 family의 **base**(교란 없음)는 scratch, 교란 케이스는 같은 family의 더 쉬운 정책에서
  **warm-start**(`USE_WARM_START=True`). **base 셀을 먼저** 실행해야 warm 원본 ckpt가 생김.
- ⚠️ 배터리 유무는 obs_dim이 달라(점유 85 vs 99) 서로 warm-start 불가 → 각 family 별도 root.
- 케이스별 `{DRIVE_ROOT}/ckpts/{EXP_TAG}_occ_<case>` 에 저장. 끊겨도 케이스 셀을 다시 실행하면 기본 `mode=resume`으로 이어서
  학습된 케이스 셀은 건너뜀.

In [ ]:
# ===== 통합 학습 공통 설정 + 헬퍼 (먼저 1번 실행) =====
%cd /content/RL-2026s1-tp
import os

USE_WARM_START  = True     # 이 파일: 교란 케이스는 전부 warm-start
COMM_FAIL_PROB  = 0.2      # 통신에러 케이스의 링크 드롭 확률 상한
FT_FRAMES, FT_LR, FT_ENT = B_FINETUNE_TOTAL_FRAMES, B_FINETUNE_LR, B_FINETUNE_ENT_COEF

# 케이스 정의 (이름 -> 조건). warm = warm-start 원본 케이스(없으면 scratch root).
CASE = {
    "nobatt_base":      dict(battery=False, comm=0.0,            rcomm=False, wind=0.0,       rwind=False, warm=None),
    "nobatt_wind":      dict(battery=False, comm=0.0,            rcomm=False, wind=WIND_PROB, rwind=True,  warm="nobatt_base"),
    "nobatt_comm":      dict(battery=False, comm=COMM_FAIL_PROB, rcomm=True,  wind=0.0,       rwind=False, warm="nobatt_base"),
    "nobatt_wind_comm": dict(battery=False, comm=COMM_FAIL_PROB, rcomm=True,  wind=WIND_PROB, rwind=True,  warm="nobatt_wind"),
    "batt_base":        dict(battery=True,  comm=0.0,            rcomm=False, wind=0.0,       rwind=False, warm=None),
    "batt_wind":        dict(battery=True,  comm=0.0,            rcomm=False, wind=WIND_PROB, rwind=True,  warm="batt_base"),
    "batt_comm":        dict(battery=True,  comm=COMM_FAIL_PROB, rcomm=True,  wind=0.0,       rwind=False, warm="batt_base"),
    "batt_wind_comm":   dict(battery=True,  comm=COMM_FAIL_PROB, rcomm=True,  wind=WIND_PROB, rwind=True,  warm="batt_wind"),
}

def case_dir(n): return f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_occ_{n}"
def case_tb(n):  return f"{DRIVE_ROOT}/runs/{EXP_TAG}_occ_{n}"

def train_case(name, mode="resume"):
    """재개 3-mode:
       'resume' (기본): 기존 pt에 '이어서' 학습. iter/frames/ckpt 번호가 이전 모델에
                        이어서 표시·저장됨(--start-iter). ckpt 없으면 새로(warm/scratch) 시작.
       'skip'         : 그 케이스에 ckpt가 있으면 건너뜀 (이미 끝난 케이스 보호).
       'restart'      : 그 케이스 폴더를 '비우고' 처음부터 (warm-start, root는 scratch)."""
    import shutil
    assert mode in ("resume", "skip", "restart"), "mode는 resume/skip/restart 중 하나"
    c = CASE[name]; sd = case_dir(name); tb = case_tb(name)
    own_ck, own_it = latest_ckpt(sd)
    if mode == "skip" and own_ck:
        print(f"[skip] {name}: 이미 ckpt 있음 ({own_ck})")
        return
    if mode == "restart":
        shutil.rmtree(sd, ignore_errors=True); own_ck, own_it = None, 0
    os.makedirs(sd, exist_ok=True)

    # 학습 강도: root(warm 없음)=full / 의존 케이스=finetune
    frames, lr, ent = (FT_FRAMES, FT_LR, FT_ENT) if c["warm"] else (TOTAL_FRAMES, LR, ENT_COEF)

    # 로드 소스: resume + 자기 ckpt 있으면 '이어서'(번호 누적), 아니면 warm 원본, 그것도 없으면 scratch
    load, start_iter, tag = "", 0, "scratch"
    if mode == "resume" and own_ck:
        load, start_iter, tag = f"--load-ckpt {own_ck}", own_it, f"resume(이어서 @iter {own_it})"
    elif USE_WARM_START and c["warm"]:
        base_ck = latest_ckpt(case_dir(c["warm"]))[0]
        assert base_ck, f"{name}: warm 원본 '{c['warm']}' ckpt 없음 → 그 케이스를 먼저 학습"
        load, tag = f"--load-ckpt {base_ck}", f"warm<-{c['warm']}"

    script = "comm_train_battery_occ.py" if c["battery"] else "comm_train_occ.py"
    batt = ""
    if c["battery"]:
        batt = (f"--initial-battery {INITIAL_BATTERY} --hover-battery-cost {HOVER_BATTERY_COST} "
                f"--move-battery-cost {MOVE_BATTERY_COST} --low-battery-move-penalty {LOW_BATTERY_MOVE_PENALTY}")
    wind = f"--wind-prob {c['wind']} --wind-strength {WIND_STRENGTH}" + (" --randomize-wind" if c["rwind"] else "")
    comm = f"--comm-fail-prob {c['comm']}" + (" --randomize-comm-fail" if c["rcomm"] else "")
    cmd = (
        f"python {script} --grid-size {GRID_SIZE} --n-agents {N_AGENTS} --max-steps {MAX_STEPS} "
        f"--shapes '{TARGET_SEQUENCE}' --completion-reward {COMPLETION_REWARD} "
        f"--assigned-target-reward {ASSIGNED_TARGET_REWARD} --coverage-delta-reward {COVERAGE_DELTA_REWARD} "
        f"--coverage-step-reward {COVERAGE_STEP_REWARD} --hover-penalty {HOVER_PENALTY} --shaping-coef {SHAPING_COEF} "
        f"{batt} {wind} {comm} "
        f"--total-frames {frames} --frames-per-batch {FRAMES_PER_BATCH} --minibatch-size {MINIBATCH_SIZE} "
        f"--ppo-epochs {PPO_EPOCHS} --lr {lr} --ent-coef {ent} --clip-eps {CLIP_EPS} "
        f"--ckpt-every {CKPT_EVERY} "
        f"--early-stop-success {EARLY_STOP_SUCCESS} --early-stop-patience {EARLY_STOP_PATIENCE} --save-best-above {SAVE_BEST_ABOVE} "
        f"--start-iter {start_iter} --save-dir {sd} --tb-logdir {tb} {load}"
    )
    print(f"===== TRAIN {name} [{tag}] | battery={c['battery']} wind={c['wind']} comm={c['comm']} =====")
    print(cmd)
    get_ipython().system(cmd)

print("train_case(name, mode='resume'(기본)|'skip'|'restart') 준비 완료. base부터 순서대로 실행.")

### 🔁 런타임이 끊겼을 때 — 이어서 학습하기 (재개 3가지 모드)

ckpt는 Drive에 저장되므로 끊겨도 이어갈 수 있습니다. `train_case(name, mode=...)`:

1. **`mode="resume"` (기본, mode 생략 시)**: 기존 pt에 **이어서** 학습.
   **iter 수·frames·ckpt 번호가 이전 모델에 이어서** 표시·저장됩니다(`--start-iter` 자동).
   ckpt가 없으면 새로(warm-start / root는 scratch) 시작 → **첫 실행에도 그대로 동작.**
2. **`mode="skip"`**: 그 케이스에 ckpt가 있으면 **건너뜀** (이미 끝난 케이스 보호).
3. **`mode="restart"`**: 그 케이스 폴더를 **비우고 처음부터** (warm-start, root는 scratch).

**재개 절차**
1. 셀 1(clone) → 2(drive) → 3(pip) → 4(설정) → 6(resume helper) → 위 **헬퍼 셀** 만 다시 실행.
2. **중간에 끊긴 케이스 셀을 그대로 다시 실행** → 기본 `resume`이라 마지막 ckpt에서 이어서 진행됩니다.
   - 이미 충분히 학습돼 더 안 돌리려면: `train_case("batt_wind", mode="skip")`
   - 완전히 새로 하려면: `train_case("batt_wind", mode="restart")`
3. warm-start **원본(base)** 이 비어 있으면 base부터 진행 후 의존 케이스를 실행하세요.

- 팁: `CKPT_EVERY`(셀 4)가 작을수록 끊겼을 때 잃는 진행분이 적습니다(기본 5 iter).
- 진행 상황 확인: `latest_ckpt(case_dir("batt_wind"))` 로 그 케이스의 마지막 ckpt를 볼 수 있음.

### 배터리 없음 (no-battery) — base 먼저, 그다음 교란 케이스

In [ ]:
train_case("nobatt_base")        # scratch (root)

In [ ]:
train_case("nobatt_wind")        # warm <- nobatt_base

In [ ]:
train_case("nobatt_comm")        # warm <- nobatt_base

In [ ]:
train_case("nobatt_wind_comm")   # warm <- nobatt_wind

### 배터리 있음 (battery) — base 먼저, 그다음 교란 케이스

In [ ]:
train_case("batt_base")        # scratch (root)

In [ ]:
train_case("batt_wind")        # warm <- batt_base

In [ ]:
train_case("batt_comm")        # warm <- batt_base

In [ ]:
train_case("batt_wind_comm")   # warm <- batt_wind

## 9. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_ROOT}/runs

## 평가 & GIF — 8개 케이스

각 케이스 ckpt를 **그 케이스가 학습된 교란 조건**(바람 `EVAL_WIND_PROB` / 통신 `COMM_FAIL_PROB` 고정)에서
평가하고, 케이스별 GIF(`demo_{EXP_TAG}_occ_<case>.gif`)와 통계 txt를 Drive에 저장합니다.
먼저 **평가 헬퍼 셀**을 1번 실행한 뒤 원하는 케이스 셀을 실행하세요.
(배터리 케이스는 `comm_eval_battery_occ.py`, 무배터리는 `comm_eval_occ.py` 자동 선택)

In [ ]:
# ===== 8개 케이스 평가 헬퍼 (먼저 1번 실행) =====
%cd /content/RL-2026s1-tp
import os
from IPython.display import Image, display

# 케이스별 최신 ckpt (학습한 것만). sweep/비교 셀 호환용 CKPT_A/B 도 세팅.
CKPTS = {n: latest_ckpt(case_dir(n))[0] for n in CASE}
print("케이스별 ckpt:")
for n in CASE:
    print(f"  {n:18s}: {CKPTS[n]}")
CKPT_A = CKPTS.get("batt_base")    # 평가 A(교란 없음) 호환
CKPT_B = CKPTS.get("batt_wind")    # 평가 B(바람) 호환

def eval_case(name, n_episodes=50, greedy=True):
    c = CASE[name]
    ck = latest_ckpt(case_dir(name))[0]            # 항상 최신 ckpt 사용
    assert ck, f"{name}: ckpt 없음 → 먼저 학습(통합 학습 셀)"
    script = "comm_eval_battery_occ.py" if c["battery"] else "comm_eval_occ.py"
    batt = ""
    if c["battery"]:
        batt = (f"--initial-battery {INITIAL_BATTERY} --hover-battery-cost {HOVER_BATTERY_COST} "
                f"--move-battery-cost {MOVE_BATTERY_COST} --low-battery-move-penalty {LOW_BATTERY_MOVE_PENALTY}")
    # 그 케이스가 학습된 교란 조건에서 평가 (바람=EVAL_WIND_PROB, 통신=COMM_FAIL_PROB 고정)
    wind = f"--wind-prob {EVAL_WIND_PROB if c['wind'] else 0.0} --wind-strength {WIND_STRENGTH}"
    comm = f"--comm-fail-prob {COMM_FAIL_PROB if c['comm'] else 0.0}"
    gif = f"{DRIVE_ROOT}/gifs/demo_{EXP_TAG}_occ_{name}.gif"
    txt = f"{DRIVE_ROOT}/evals/eval_{EXP_TAG}_occ_{name}.txt"
    g = "--greedy" if greedy else ""
    cmd = (f"python {script} --ckpt {ck} --grid-size {GRID_SIZE} --n-agents {N_AGENTS} --max-steps {MAX_STEPS} "
           f"--shapes '{TARGET_SEQUENCE}' --completion-reward {COMPLETION_REWARD} {batt} {wind} {comm} "
           f"{g} --n-episodes {n_episodes} --save-gif {gif} --out {txt}")
    print(f"===== EVAL {name} | battery={c['battery']} wind={c['wind']>0} comm={c['comm']>0} =====")
    get_ipython().system(cmd)
    get_ipython().system(f"cat {txt}")

def show_gif(name):
    g = f"{DRIVE_ROOT}/gifs/demo_{EXP_TAG}_occ_{name}.gif"
    if os.path.exists(g):
        print(f"=== {name} ==="); display(Image(g))
    else:
        print(f"{name}: GIF 없음 (먼저 eval_case('{name}'))")

print("\neval_case(name) / show_gif(name) 준비됨. 아래 케이스 셀을 실행하세요.")

### 배터리 없음 — 평가

In [ ]:
eval_case("nobatt_base")

In [ ]:
eval_case("nobatt_wind")

In [ ]:
eval_case("nobatt_comm")

In [ ]:
eval_case("nobatt_wind_comm")

### 배터리 있음 — 평가

In [ ]:
eval_case("batt_base")

In [ ]:
eval_case("batt_wind")

In [ ]:
eval_case("batt_comm")

In [ ]:
eval_case("batt_wind_comm")

### GIF 표시 (학습/평가된 케이스 전부)

In [ ]:
# 학습/평가된 모든 케이스의 GIF 표시
for n in CASE:
    show_gif(n)

### 📊 요약 표 — 8개 케이스 성공률 / coverage

학습된 케이스들을 **각자의 학습 교란 조건**(바람 `EVAL_WIND_PROB` / 통신 `COMM_FAIL_PROB` 고정)에서
다시 굴려 한 표로 비교합니다. `comm_eval_strict.rollout`을 써서 종료 사유를 분류 → **freeze는
`timeout` 비율**로, 충돌은 `collision`으로 나타납니다. 미학습 케이스는 자동 건너뜀.

In [ ]:
# ===== 8개 케이스 성공률/coverage 요약 표 =====
%cd /content/RL-2026s1-tp
import numpy as np, torch
from collections import Counter
from torchrl.envs.utils import ExplorationType
import comm_eval_strict as ces
from comm_env_occupancy import OccupancyShapeFormationEnv, BatteryOccupancyShapeFormationEnv

SUMMARY_N_EPISODES = 50
SUMMARY_GREEDY     = True     # True=greedy(평가와 동일) / False=stochastic(분포)

ces.ShapeFormationEnv = OccupancyShapeFormationEnv
ces.BatteryShapeFormationEnv = BatteryOccupancyShapeFormationEnv
shapes_sum = [s.strip() for s in TARGET_SEQUENCE.split(',') if s.strip()]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def _case_metrics(name):
    c = CASE[name]; ck = latest_ckpt(case_dir(name))[0]
    if not ck:
        return None
    env, base = ces.make_env(
        seed=0, device=device, use_battery=c["battery"],
        grid_size=GRID_SIZE, n_agents=N_AGENTS, max_steps=MAX_STEPS, shapes=shapes_sum,
        comm_fail_prob=(COMM_FAIL_PROB if c["comm"] else 0.0),
        completion_reward=COMPLETION_REWARD,
        wind_prob=(EVAL_WIND_PROB if c["wind"] else 0.0),
        wind_strength=WIND_STRENGTH, randomize_wind=False,
        initial_battery=INITIAL_BATTERY, hover_battery_cost=HOVER_BATTERY_COST,
        move_battery_cost=MOVE_BATTERY_COST, low_battery_move_penalty=LOW_BATTERY_MOVE_PENALTY,
    )
    actor = ces.build_actor(base.obs_dim, 5, base.n_agents, 128, device)
    with torch.no_grad():
        actor(env.reset())
    actor.load_state_dict(torch.load(ck, map_location=device)["actor"]); actor.eval()
    expl = ExplorationType.MODE if SUMMARY_GREEDY else ExplorationType.RANDOM
    runs = [ces.rollout(env, base, actor, expl, max_steps=base.max_steps) for _ in range(SUMMARY_N_EPISODES)]
    n = len(runs); reasons = Counter(r["end_reason"] for r in runs)
    return dict(
        success=sum(r["success"] for r in runs) / n,
        timeout=reasons.get("timeout", 0) / n,
        collision=reasons.get("collision", 0) / n,
        battery=reasons.get("battery_depleted", 0) / n,
        coverage=float(np.mean([r["coverage"] for r in runs])),
        steps=float(np.mean([r["steps"] for r in runs])),
    )

rows = []
for name in CASE:
    m = _case_metrics(name)
    rows.append((name, m))
    print(("[done] " if m else "[skip:no ckpt] ") + name)

def col(v, w, pct=False):
    return (f"{v:.1%}" if pct else f"{v:.1f}").rjust(w)

print(f"\n=== 8개 케이스 요약 (greedy={SUMMARY_GREEDY}, n={SUMMARY_N_EPISODES}, 각 케이스 학습 교란 조건) ===")
hdr = ("case".ljust(18) + "success".rjust(9) + "timeout".rjust(9) + "collision".rjust(10)
       + "battery".rjust(9) + "coverage".rjust(10) + "steps".rjust(8))
print(hdr); print("-" * len(hdr))
for name, m in rows:
    if m is None:
        print(name.ljust(18) + "n/a".rjust(9)); continue
    print(name.ljust(18) + col(m["success"], 9, True) + col(m["timeout"], 9, True)
          + col(m["collision"], 10, True) + col(m["battery"], 9, True)
          + col(m["coverage"], 10, True) + col(m["steps"], 8))

## 통합 robustness 비교 — 정책 × 평가조건 (배터리·바람·통신 한 번에)

바람만 보는 sweep 대신, **학습된 모든 케이스 정책(행)** 을 **여러 평가조건(열: clean / 바람 / 통신 / 바람+통신)**
에서 교차 평가해 한 표(+히트맵)로 비교합니다.

- **배터리 차원**: 정책(행)이 `batt_*` / `nobatt_*` 로 구분 (배터리 정책은 배터리 env, 무배터리는 무배터리 env에서 평가).
- **바람/통신 차원**: 평가조건(열)로 부여 (`EVAL_WIND_PROB` / `COMM_FAIL_PROB` 고정).
- 셀 값 = 성공률(%). 대각선 근처(자기 학습조건)뿐 아니라 **off-조건에서의 일반화**까지 보여줍니다.
  (예: comm으로 학습한 정책이 바람에서도 버티는가, base가 교란에서 무너지는가)
- 학습된 케이스만 자동 포함. `XEVAL_N`(에피소드 수)으로 속도/정밀도 조절.

In [ ]:
# ===== 통합 robustness 교차평가: 정책(8케이스) x 평가조건 =====
%cd /content/RL-2026s1-tp
import numpy as np, torch
from torchrl.envs.utils import ExplorationType
import comm_eval_strict as ces
from comm_env_occupancy import OccupancyShapeFormationEnv, BatteryOccupancyShapeFormationEnv

XEVAL_N        = 30        # 조건당 에피소드 수 (8정책 x 4조건 x N → 시간 고려)
XEVAL_GREEDY   = True
XEVAL_POLICIES = None      # None=학습된 케이스 전부 / 또는 ["batt_base","batt_wind",...]

ces.ShapeFormationEnv = OccupancyShapeFormationEnv
ces.BatteryShapeFormationEnv = BatteryOccupancyShapeFormationEnv
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
shapes_x = [s.strip() for s in TARGET_SEQUENCE.split(',') if s.strip()]

# 평가조건(열): (이름, wind_prob, comm_fail_prob)
CONDS = [("clean", 0.0, 0.0),
         ("wind", EVAL_WIND_PROB, 0.0),
         ("comm", 0.0, COMM_FAIL_PROB),
         ("wind+comm", EVAL_WIND_PROB, COMM_FAIL_PROB)]

policies = XEVAL_POLICIES or [n for n in CASE if latest_ckpt(case_dir(n))[0]]
assert policies, "학습된 케이스 ckpt가 없음 → 먼저 통합 학습 셀 실행"

def _run(name, wind, comm):
    c = CASE[name]; ck = latest_ckpt(case_dir(name))[0]
    env, base = ces.make_env(
        seed=0, device=device, use_battery=c["battery"],
        grid_size=GRID_SIZE, n_agents=N_AGENTS, max_steps=MAX_STEPS, shapes=shapes_x,
        comm_fail_prob=comm, completion_reward=COMPLETION_REWARD,
        wind_prob=wind, wind_strength=WIND_STRENGTH, randomize_wind=False,
        initial_battery=INITIAL_BATTERY, hover_battery_cost=HOVER_BATTERY_COST,
        move_battery_cost=MOVE_BATTERY_COST, low_battery_move_penalty=LOW_BATTERY_MOVE_PENALTY,
    )
    actor = ces.build_actor(base.obs_dim, 5, base.n_agents, 128, device)
    with torch.no_grad():
        actor(env.reset())
    actor.load_state_dict(torch.load(ck, map_location=device)["actor"]); actor.eval()
    expl = ExplorationType.MODE if XEVAL_GREEDY else ExplorationType.RANDOM
    runs = [ces.rollout(env, base, actor, expl, max_steps=base.max_steps) for _ in range(XEVAL_N)]
    return (np.mean([r["success"] for r in runs]), np.mean([r["coverage"] for r in runs]))

succ = np.zeros((len(policies), len(CONDS)))
cov  = np.zeros((len(policies), len(CONDS)))
for i, name in enumerate(policies):
    for j, (cn, w, cm) in enumerate(CONDS):
        succ[i, j], cov[i, j] = _run(name, w, cm)
        print(f"  {name:18s} @ {cn:10s}: success={succ[i,j]:.0%} coverage={cov[i,j]:.0%}")

def _table(title, M):
    print(f"\n=== {title} (행=정책, 열=평가조건, n={XEVAL_N}) ===")
    hdr = "policy".ljust(18) + "".join(c[0].rjust(11) for c in CONDS)
    print(hdr); print("-" * len(hdr))
    for i, name in enumerate(policies):
        print(name.ljust(18) + "".join(f"{M[i,j]*100:10.0f}%" for j in range(len(CONDS))))

_table("성공률(%)", succ)
_table("coverage(%)", cov)

# 히트맵 PNG (성공률) — Drive 저장 + 표시
import matplotlib
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(1.8 + 1.2 * len(CONDS), 1.0 + 0.5 * len(policies)))
im = ax.imshow(succ, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(CONDS))); ax.set_xticklabels([c[0] for c in CONDS])
ax.set_yticks(range(len(policies))); ax.set_yticklabels(policies)
for i in range(len(policies)):
    for j in range(len(CONDS)):
        ax.text(j, i, f"{succ[i,j]*100:.0f}", ha="center", va="center", fontsize=8)
ax.set_title(f"success rate (%): policy x eval condition  [{TARGET_SEQUENCE}]")
fig.colorbar(im, ax=ax, label="success")
png = f"{DRIVE_ROOT}/sweeps/xeval_{EXP_TAG}_occ.png"
fig.tight_layout(); fig.savefig(png, dpi=120); plt.close(fig)
from IPython.display import Image, display
print("\nsaved heatmap ->", png); display(Image(png))

## 10. Drive 산출물 점검

In [ ]:
print("=== occ ckpt 폴더 ===")
!ls -la {DRIVE_ROOT}/ckpts/ | grep occ || true
print("\n=== gifs (occ) ==="); !ls -la {DRIVE_ROOT}/gifs/ 2>/dev/null | grep occ || true
print("\n=== evals (occ) ==="); !ls -la {DRIVE_ROOT}/evals/ 2>/dev/null | grep occ || true

## 11. 비교 실험 — 비-점유(base) vs 점유(occupancy)

같은 시퀀스/조건에서 두 정책을 평가해 **success / timeout(=freeze) / collision /
평균 coverage**를 표로 비교합니다. `comm_eval_strict.rollout`을 써서 종료 사유를
분류하므로, **얼어붙음은 `timeout` 비율**로 직접 드러납니다.

- 비-점유 baseline ckpt는 `colab_runner_final.ipynb`로 **같은 `EXP_TAG`** 를 학습해
  `{DRIVE_ROOT}/ckpts/{EXP_TAG}_battery` 에 있어야 함. 없으면 occupancy만 출력.
- `CMP_GREEDY=False`(기본)면 stochastic으로 분포 확보. `CMP_WIND_PROB`/`CMP_COMM_FAIL`을
  올리면 교란 하 robustness 비교가 됨.

In [ ]:
# ===== 비교 실험: 비-점유(base) vs 점유(occupancy) =====
%cd /content/RL-2026s1-tp
import numpy as np, torch
from collections import Counter
from torchrl.envs.utils import ExplorationType
import comm_eval_strict as ces
from comm_env import BatteryShapeFormationEnv
from comm_env_occupancy import BatteryOccupancyShapeFormationEnv

CMP_N_EPISODES = 100
CMP_GREEDY     = False     # False=stochastic(분포) / True=greedy(결정적)
CMP_WIND_PROB  = 0.0       # >0 이면 바람 속 비교
CMP_COMM_FAIL  = 0.0       # >0 이면 통신두절 속 비교

shapes_cmp = [s.strip() for s in TARGET_SEQUENCE.split(',') if s.strip()]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE_SAVE_DIR_A = f"{DRIVE_ROOT}/ckpts/{EXP_TAG}_battery"   # 비-점유 (final 러너)
OCC_SAVE_DIR_A  = SAVE_DIR_A                                # 점유 (이 러너)

def _run(env_class, ckpt):
    ces.BatteryShapeFormationEnv = env_class               # env 클래스만 교체
    env, base = ces.make_env(
        seed=0, device=device, use_battery=True,
        grid_size=GRID_SIZE, n_agents=N_AGENTS, max_steps=MAX_STEPS,
        shapes=shapes_cmp, comm_fail_prob=CMP_COMM_FAIL,
        completion_reward=COMPLETION_REWARD, wind_prob=CMP_WIND_PROB,
        wind_strength=WIND_STRENGTH, randomize_wind=False,
        initial_battery=INITIAL_BATTERY, hover_battery_cost=HOVER_BATTERY_COST,
        move_battery_cost=MOVE_BATTERY_COST, low_battery_move_penalty=LOW_BATTERY_MOVE_PENALTY,
    )
    actor = ces.build_actor(base.obs_dim, 5, base.n_agents, 128, device)
    with torch.no_grad():
        actor(env.reset())
    state = torch.load(ckpt, map_location=device)
    actor.load_state_dict(state['actor'])
    actor.eval()
    expl = ExplorationType.MODE if CMP_GREEDY else ExplorationType.RANDOM
    return [ces.rollout(env, base, actor, expl, max_steps=base.max_steps)
            for _ in range(CMP_N_EPISODES)]

def _metrics(runs):
    n = len(runs); reasons = Counter(r['end_reason'] for r in runs)
    return {
        'success':   sum(r['success'] for r in runs) / n,
        'timeout':   reasons.get('timeout', 0) / n,
        'collision': reasons.get('collision', 0) / n,
        'battery':   reasons.get('battery_depleted', 0) / n,
        'coverage':  float(np.mean([r['coverage'] for r in runs])),
        'steps':     float(np.mean([r['steps'] for r in runs])),
    }

base_ck, _ = latest_ckpt(BASE_SAVE_DIR_A)
occ_ck,  _ = latest_ckpt(OCC_SAVE_DIR_A)
print(f"base(no occ) ckpt: {base_ck}")
print(f"occupancy    ckpt: {occ_ck}")

results = {}
if base_ck:
    results['base (no occ)'] = _metrics(_run(BatteryShapeFormationEnv, base_ck))
else:
    print("⚠️ 비-점유 baseline 없음 → colab_runner_final.ipynb 로 같은 EXP_TAG 학습 필요.")
if occ_ck:
    results['occupancy'] = _metrics(_run(BatteryOccupancyShapeFormationEnv, occ_ck))
else:
    print("⚠️ occupancy ckpt 없음 → 셀 8 학습 먼저.")

if results:
    cond = f"greedy={CMP_GREEDY}, wind={CMP_WIND_PROB}, comm_fail={CMP_COMM_FAIL}, n={CMP_N_EPISODES}"
    print(f"\n=== 비교 ({cond}) ===")
    cols = list(results.keys())
    pct = {'success','timeout','collision','battery','coverage'}
    rows = [('success rate','success'), ('timeout(freeze) rate','timeout'),
            ('collision rate','collision'), ('battery-dead rate','battery'),
            ('mean coverage','coverage'), ('mean steps','steps')]
    hdr = f"{'metric':<22}" + "".join(f"{c:>16}" for c in cols)
    print(hdr); print('-' * len(hdr))
    for label, key in rows:
        line = f"{label:<22}"
        for c in cols:
            v = results[c][key]
            line += f"{(f'{v:.1%}' if key in pct else f'{v:.1f}'):>16}"
        print(line)

---
## 메모

- **학습은 '통합 학습' 섹션의 케이스별 셀에서** 원하는 조합만 골라 실행합니다.
  (배터리 단독 A/B 학습 셀은 매트릭스의 `batt_base`/`batt_wind`와 중복이라 제거됨)
- 평가/비교 셀의 A/B는 각각 `batt_base`(배터리·교란없음), `batt_wind`(배터리+바람)
  케이스 ckpt를 가리킵니다(config 셀에서 경로 지정).
- 점유 관측은 obs_dim을 바꾸므로 기존(비-점유) ckpt와 호환되지 않습니다. 반드시 처음부터 학습.
- 비교 실험: 같은 시퀀스를 `colab_runner_final.ipynb`(비-점유)와 이 노트북(점유)으로
  각각 학습해 GIF/성공률을 비교하면 점유 관측이 교착(freeze)을 얼마나 줄이는지 확인 가능.
- 변경 상세 논리는 저장소 루트의 `OCCUPANCY_CHANGES.txt` 참고.